# Bab 12 - Neural Networks that Write Like Shakespeare: Recurrent Layers

**Nama:** Nanda Pratama ITM  
**NIM:** 101032330212  
**Kelas:** TK 47 01


> Notebook ini merupakan reproduksi edukatif dan pendalaman teori berdasarkan *Grokking Deep Learning* (Andrew W. Trask, Manning, 2019). Kode inti diadaptasi dari buku dan repositori pendamping resmi; perubahan kompatibilitas dijelaskan dalam dokumentasi.


## Capaian pembelajaran

Setelah menyelesaikan notebook ini, pembaca diharapkan mampu menjelaskan konsep utama bab, menghubungkannya dengan operasi NumPy, menjalankan contoh secara berurutan, dan menyebutkan batas interpretasinya.


## Data dengan panjang berubah

RNN memproses token secara berurutan dan membawa hidden state. Bobot transisi yang sama dipakai pada setiap langkah sehingga model dapat menerima panjang sequence berbeda.


## Bag of words vs. recurrence

Rata-rata embedding mengabaikan urutan. RNN memperbarui keadaan $h_t=f(h_{t-1}W_h+x_tW_x)$ sehingga urutan dapat memengaruhi representasi dan prediksi berikutnya.


## Backpropagation through time

BPTT membuka RNN sepanjang waktu lalu menerapkan chain rule. Gradien dari langkah akhir mengalir melalui banyak transisi; komputasi dan memori bertambah bersama panjang sequence.


## Perplexity

Perplexity adalah eksponensial rata-rata negative log-likelihood per token. Lebih kecil berarti model memberi probabilitas lebih tinggi pada token benar, tetapi perbandingan hanya adil pada tokenisasi dan dataset yang sama.


## Reproduksi kode dan eksperimen

Jalankan sel dari atas ke bawah. `MODE_CEPAT=True` menjaga waktu eksekusi singkat; ubah ke `False` untuk konfigurasi yang lebih panjang.


In [1]:
from pathlib import Path
import numpy as np

np.set_printoptions(precision=4, suppress=True)
np.random.seed(1)
DATA_DIR = (Path.cwd().parent / "data").resolve()
MODE_CEPAT = True  # ubah ke False untuk eksperimen lebih panjang
assert DATA_DIR.is_dir(), f"Folder data tidak ditemukan: {DATA_DIR}"
print("Data:", DATA_DIR)
print("Mode cepat:", MODE_CEPAT)

def sigmoid(x):
    x = np.clip(x, -60, 60)
    return 1 / (1 + np.exp(-x))

def softmax(x):
    x = np.asarray(x)
    shifted = x - np.max(x, axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=-1, keepdims=True)


Data: D:\Kuliah\DL\UPLOAD_GITHUB\Grokking-Deep-Learning\data
Mode cepat: True


In [2]:
raw = (DATA_DIR / "qa1_single-supporting-fact_train.txt").read_text(encoding="utf-8").splitlines()
sentences = [line.lower().split()[1:] for line in raw if "?" not in line][: (300 if MODE_CEPAT else 1000)]
vocab = sorted({word.strip(".?") for sent in sentences for word in sent})
word2index = {word: i for i, word in enumerate(vocab)}
sequences = [[word2index[w.strip(".?")] for w in sent] for sent in sentences]
print("sentences/vocab:", len(sequences), len(vocab), sentences[0])


sentences/vocab: 300 17 ['mary', 'moved', 'to', 'the', 'bathroom.']


In [3]:
rng = np.random.default_rng(12)
vocab_size, hidden_size = len(vocab), 16
embed = rng.normal(0, 0.08, (vocab_size, hidden_size))
recurrent = np.eye(hidden_size)
decoder = rng.normal(0, 0.08, (hidden_size, vocab_size))
alpha, steps = 0.02, (800 if MODE_CEPAT else 5000)

for step in range(steps):
    seq = sequences[step % len(sequences)]
    states = [np.zeros(hidden_size)]
    probs = []
    for token in seq[:-1]:
        states.append(np.tanh(states[-1] @ recurrent + embed[token]))
        probs.append(softmax(states[-1] @ decoder))
    dh_next = np.zeros(hidden_size)
    for t in reversed(range(len(probs))):
        dlogit = probs[t].copy(); dlogit[seq[t+1]] -= 1
        decoder -= alpha * np.outer(states[t+1], dlogit)
        dh = dlogit @ decoder.T + dh_next
        dz = dh * (1 - states[t+1] ** 2)
        recurrent -= alpha * np.outer(states[t], dz)
        embed[seq[t]] -= alpha * dz
        dh_next = dz @ recurrent.T

losses = []
for seq in sequences[:50]:
    h = np.zeros(hidden_size)
    for current, target in zip(seq[:-1], seq[1:]):
        h = np.tanh(h @ recurrent + embed[current])
        losses.append(-np.log(softmax(h @ decoder)[target] + 1e-12))
print("perplexity:", float(np.exp(np.mean(losses))))


perplexity: 2.5288067051472054


## Interpretasi dan kesimpulan

Hasil harus dibaca bersama asumsi, konfigurasi, seed, serta data yang digunakan. Keberhasilan menjalankan kode tidak membuktikan generalisasi, kausalitas, keamanan, atau kelayakan deployment. Bandingkan hasil dengan teori di atas dan jelaskan setiap perbedaan sebelum mengubah konfigurasi.
